In [ ]:
#WE BEGIN BY CREATING OUR SYNTHETIC DATASET AND TRAINING THE FIRST NEURAL NETWORK

import random
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
import torch.optim.lr_scheduler as lr_scheduler

# ==========================================
# 0. CONFIGURATION
# ==========================================
APPLY_NOISE = False       
NOISE_BASE = 0.2         # Base minimum noise
NOISE_FACTOR = 0.15      # Noise scaling based on operation weight
DATASET_SIZE = 10000     
EPOCHS = 100             

# ==========================================
# 1. DATASET GENERATION
# ==========================================
def calculate_sequence_costs(sequence, apply_noise=True, noise_base=0.2, noise_factor=0.15):
    """
    Calculates the cost of a sequence. If apply_noise is True, it simulates 
    stochastic delays or advances based on the provided noise parameters.
    """
    total_clean_cost = 0.0     # The pure mathematical sum
    total_noisy_cost = 0.0     # The actual clocked time (with or without noise)
    previous = None
    
    for char in sequence:
        # 1. Determine the theoretical mean (mu) of this transition
        if char == 'A':
            mu = 1.0
        elif char == 'B':
            mu = 0.0 if previous == 'A' else 2.0
        elif char == 'C':
            mu = 10.0 if previous == 'C' else 5.0
            
        # 2. Apply noise if enabled
        if apply_noise:
            sigma = noise_base + (noise_factor * mu)
            operation_time = random.gauss(mu, sigma)
            operation_time = max(0.0, operation_time) # No negative time
        else:
            operation_time = mu
        
        # 3. Accumulate
        total_clean_cost += mu
        total_noisy_cost += operation_time
        previous = char
        
    return total_noisy_cost, total_clean_cost

data = []
for _ in range(DATASET_SIZE):
    length = random.randint(1, 10)
    sequence = "".join(random.choices(['A', 'B', 'C'], k=length))
    
    # Get costs using the defined configuration
    c_noisy, c_clean = calculate_sequence_costs(
        sequence, 
        apply_noise=APPLY_NOISE, 
        noise_base=NOISE_BASE, 
        noise_factor=NOISE_FACTOR
    )
    
    data.append({
        'sequence': sequence, 
        'target_cost': c_noisy,  # RNN Target (will be clean if noise is False)
        'clean_real_cost': c_clean 
    })

df = pd.DataFrame(data)

print(f"Dataset Sample (Noise Applied: {APPLY_NOISE}):")
# Use print(df.head(7)) if you are running this in a standard terminal
try:
    display(df.head(7))
except NameError:
    print(df.head(7))

# ==========================================
# 2. DATA PREPARATION
# ==========================================
vocab = {'A': 1, 'B': 2, 'C': 3}

class StepSequenceDataset(Dataset):
    def __init__(self, dataframe):
        self.X = [torch.tensor([vocab[char] for char in seq], dtype=torch.long) for seq in dataframe['sequence']]
        self.Y = torch.tensor(dataframe['target_cost'].values, dtype=torch.float32)

    def __len__(self): return len(self.X)
    def __getitem__(self, idx): return self.X[idx], self.Y[idx]

def collate_fn(batch):
    X, Y = zip(*batch)
    lengths = torch.tensor([len(x) for x in X], dtype=torch.long)
    X_pad = pad_sequence(X, batch_first=True, padding_value=0)
    return X_pad, torch.stack(Y), lengths

dataset = StepSequenceDataset(df)
train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size
train_dataset, test_dataset = torch.utils.data.random_split(dataset, [train_size, test_size])

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, collate_fn=collate_fn)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, collate_fn=collate_fn)

# ==========================================
# 3. MODEL 
# ==========================================
class CostPredictorRNN(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super(CostPredictorRNN, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.rnn = nn.RNN(embed_dim, hidden_dim, batch_first=True, nonlinearity='relu')
        self.linear = nn.Linear(hidden_dim, 1)

    def forward(self, x, lengths):
        emb = self.embedding(x)
        packed_emb = nn.utils.rnn.pack_padded_sequence(emb, lengths, batch_first=True, enforce_sorted=False)
        _, h_n = self.rnn(packed_emb)
        prediction = self.linear(h_n[-1, :, :])
        return prediction.squeeze()

model = CostPredictorRNN(vocab_size=4, embed_dim=8, hidden_dim=32)
criterion = nn.L1Loss() 
optimizer = optim.Adam(model.parameters(), lr=0.01)
scheduler = lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5, verbose=True)

# ==========================================
# 4. TRAINING
# ==========================================
print("\nStarting training...")

for epoch in range(EPOCHS):
    model.train()
    total_train_loss = 0
    for X_batch, Y_batch, lengths in train_loader:
        optimizer.zero_grad()
        predictions = model(X_batch, lengths)
        if predictions.dim() == 0: predictions = predictions.unsqueeze(0)
        loss = criterion(predictions, Y_batch)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_train_loss += loss.item()
    
    avg_train_loss = total_train_loss / len(train_loader)
    
    model.eval()
    total_test_loss = 0
    with torch.no_grad():
        for X_batch_test, Y_batch_test, lengths_test in test_loader:
            test_predictions = model(X_batch_test, lengths_test)
            if test_predictions.dim() == 0: test_predictions = test_predictions.unsqueeze(0)
            test_loss = criterion(test_predictions, Y_batch_test)
            total_test_loss += test_loss.item()
            
    avg_test_loss = total_test_loss / len(test_loader)
    scheduler.step(avg_test_loss)
    
    if (epoch + 1) % 10 == 0 or epoch == 0:
        current_lr = optimizer.param_groups[0]['lr']
        print(f"Epoch {epoch+1:03d}/{EPOCHS} - Train MAE: {avg_train_loss:.4f} | Test MAE: {avg_test_loss:.4f} | LR: {current_lr:.6f}")

# ==========================================
# 5. TESTING
# ==========================================
print("\nTesting predictions (Network vs Reality):")
model.eval()
examples = ["A", "AB", "ABC", "CCC", "CCB", "AAB", "CBACBACBA"]

with torch.no_grad():
    for seq in examples:
        tensor_in = torch.tensor([vocab[c] for c in seq], dtype=torch.long).unsqueeze(0)
        length = torch.tensor([len(seq)], dtype=torch.long)
        
        pred = model(tensor_in, length).item()
        
        # Get the real clean cost for comparison
        _, clean_real = calculate_sequence_costs(seq, apply_noise=False)
        
        print(f"Sequence: {seq:<9} | Truth: {clean_real:5.1f} | RNN Prediction: {pred:5.1f}")

In [ ]:
#WE LEARN THE NWFA FROM THE NETWORK

import torch
import sys
import os

from QuantitativeObservationTable import (
    QuantitativeObservationTable, 
    QuantitativeObservationTableParameters
)

from TrainedRNNOracle import TrainedRNNOracle

oracle = TrainedRNNOracle(rnn_model=model, vocab_dict=vocab, tol=0.5, rounding_step=1)

params = QuantitativeObservationTableParameters(
    tol_dist_init=0.7, 
    rec_method="spectral", 
    r=3, 
)

table = QuantitativeObservationTable(
    alphabet=oracle.alphabet,
    membership_query=oracle.calculate_weight, 
    parameters=params
)

seen_counterexamples = set()
iteration = 1

while True:
    print(f"\n--- Iteration {iteration} ---")
    table.make_table_closed_and_consistent()
            
    # Reconstruct Hypothesis WFA
    wfa = table.reconstruct_WFA()
    wfa.check_twins_property()
    
    # Equivalence Query
    ce = oracle.equivalence_query(
        wfa, 
        method="hybrid", 
        exhaustive_len=6, 
        random_max_len=15, 
        num_random=1000,
    )

    # Evaluate Equivalence Query Results
    if ce is None:
        print(f"\n[SUCCESS] No counterexamples found.")
        print(f"Extraction complete in {iteration} iterations. Total states: {len(table.S)}")
        break

    if ce in seen_counterexamples:
        print(f"\n[STOP] Repeated counterexample '{ce}' detected. Check tolerances or network precision.")
        break

    print(f"[+] Processing Counterexample: '{ce}' | S size: {len(table.S)}")

    # Process Counterexample
    table.process_counterexample(ce, hypothesis_wfa=wfa, method="all_suffixes")
    seen_counterexamples.add(ce)
            
    iteration += 1

In [ ]:
#FUNCTION FOR DRAWING THE LEARNED WFAs

from graphviz import Digraph
import numpy as np

def draw_tropical_wfa(wfa: 'TropicalWFA', name="TropicalWFA"):
    """
    Generates a Graphviz diagram for a TropicalWFA object,
    formatting weights to a maximum of 3 decimal places.
    """
    dot = Digraph(name)
    dot.attr(rankdir='LR') # Left-to-right orientation
    
    n_states = len(wfa.q0)
    
    # 1. Create nodes (states) and initial/final transitions
    for i in range(n_states):
        label = f"S{i}"
        
        # Process final weights (beta)
        is_final = wfa.final[i] != float('-inf')
        if is_final:
            # Se aplica round() para limitar a 3 decimales
            label += f"\n[β={round(wfa.final[i], 3)}]"
            # Double circle for final states
            dot.node(str(i), label, shape="doublecircle")
        else:
            # Normal circle for the rest
            dot.node(str(i), label, shape="circle")
        
        # Process initial weights (alpha) - Incoming arrows
        if wfa.q0[i] != float('-inf'):
            # Invisible ghost node for the incoming arrow
            dot.node(f"start{i}", "", shape="none", width="0", height="0")
            # Se aplica round() al peso inicial
            dot.edge(f"start{i}", str(i), label=f"α={round(wfa.q0[i], 3)}")

    # 2. Create internal transitions (delta)
    for char in wfa.alphabet:
        matrix = wfa.delta[char]
        for i in range(n_states):
            for j in range(n_states):
                weight = matrix[i, j]
                # If the weight is not -inf, there is a transition
                if weight != float('-inf'):
                    # Se aplica round() al peso de la transición
                    dot.edge(str(i), str(j), label=f"{char}:{round(weight, 3)}")
                    
    return dot



In [ ]:
#WE DRAW OUR LEARNED NWFA

dot = draw_tropical_wfa(wfa)
dot.render('NoDetNoNoiseCase1', view=True,format='png')

In [ ]:
#WE LEARN A DWFA

import torch
import sys
import os

from QuantitativeObservationTable import (
    QuantitativeObservationTable, 
    QuantitativeObservationTableParameters
)

from TrainedRNNOracle import TrainedRNNOracle

oracle = TrainedRNNOracle(rnn_model=model, vocab_dict=vocab, tol=0.5, rounding_step=1)

params = QuantitativeObservationTableParameters(
    tol_dist_init=0.7, 
    rec_method="direct", 
    r=3, 
)

table = QuantitativeObservationTable(
    alphabet=oracle.alphabet,
    membership_query=oracle.calculate_weight, 
    parameters=params
)

seen_counterexamples = set()
iteration = 1

while True:
    print(f"\n--- Iteration {iteration} ---")
    table.make_table_closed_and_consistent_det()
            
    # Reconstruct Hypothesis WFA
    wfa = table.reconstruct_deterministic_wfa_opt()
    wfa.check_twins_property()
    
    # Equivalence Query
    ce = oracle.equivalence_query(
        wfa, 
        method="hybrid", 
        exhaustive_len=6, 
        random_max_len=15, 
        num_random=1000,
    )

    # Evaluate Equivalence Query Results
    if ce is None:
        print(f"\n[SUCCESS] No counterexamples found.")
        print(f"Extraction complete in {iteration} iterations. Total states: {len(table.S)}")
        break

    if ce in seen_counterexamples:
        print(f"\n[STOP] Repeated counterexample '{ce}' detected. Check tolerances or network precision.")
        break

    print(f"[+] Processing Counterexample: '{ce}' | S size: {len(table.S)}")

    # Process Counterexample
    table.process_counterexample(ce, hypothesis_wfa=wfa, method="all_suffixes")
    seen_counterexamples.add(ce)
            
    iteration += 1

In [ ]:
#WE MANIPULATE THE WEIGHTS AS DESCRIBED IN THE THESIS

wfa.push_weights_to_positive()
wfa.normalize_global_bias()
wfa.push_final_weights_to_zero()

In [ ]:
#WE DRAW OUR DWFA

dot = draw_tropical_wfa(wfa)
dot.render('DetProc3Case1', view=True,format='png')

In [ ]:
#WE TRAIN AN RNN USING ONLY SEQUENCES OF LENGTH GREATER OR EQUAL TO 3 (BIASED RNN)

import random
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
import torch.optim.lr_scheduler as lr_scheduler

# ==========================================
# 0. CONFIGURATION
# ==========================================
APPLY_NOISE = False      
NOISE_BASE = 0.2         # Base minimum noise
NOISE_FACTOR = 0.15      # Noise scaling based on operation weight
DATASET_SIZE = 10000     
EPOCHS = 100             

# ==========================================
# 1. DATASET GENERATION
# ==========================================
def calculate_sequence_costs(sequence, apply_noise=True, noise_base=0.2, noise_factor=0.15):
    """
    Calculates the cost of a sequence. If apply_noise is True, it simulates 
    stochastic delays or advances based on the provided noise parameters.
    """
    total_clean_cost = 0.0     # The pure mathematical sum
    total_noisy_cost = 0.0     # The actual clocked time (with or without noise)
    previous = None
    
    for char in sequence:
        # 1. Determine the theoretical mean (mu) of this transition
        if char == 'A':
            mu = 1.0
        elif char == 'B':
            mu = 0.0 if previous == 'A' else 2.0
        elif char == 'C':
            mu = 10.0 if previous == 'C' else 5.0
            
        # 2. Apply noise if enabled
        if apply_noise:
            sigma = noise_base + (noise_factor * mu)
            operation_time = random.gauss(mu, sigma)
            operation_time = max(0.0, operation_time) # No negative time
        else:
            operation_time = mu
        
        # 3. Accumulate
        total_clean_cost += mu
        total_noisy_cost += operation_time
        previous = char
        
    return total_noisy_cost, total_clean_cost

data = []
for _ in range(DATASET_SIZE):
    length = random.randint(3, 10)
    sequence = "".join(random.choices(['A', 'B', 'C'], k=length))
    
    # Get costs using the defined configuration
    c_noisy, c_clean = calculate_sequence_costs(
        sequence, 
        apply_noise=APPLY_NOISE, 
        noise_base=NOISE_BASE, 
        noise_factor=NOISE_FACTOR
    )
    
    data.append({
        'sequence': sequence, 
        'target_cost': c_noisy,  # RNN Target (will be clean if noise is False)
        'clean_real_cost': c_clean 
    })

df = pd.DataFrame(data)

print(f"Dataset Sample (Noise Applied: {APPLY_NOISE}):")
# Use print(df.head(7)) if you are running this in a standard terminal
try:
    display(df.head(7))
except NameError:
    print(df.head(7))

# ==========================================
# 2. DATA PREPARATION
# ==========================================
vocab = {'A': 1, 'B': 2, 'C': 3}

class StepSequenceDataset(Dataset):
    def __init__(self, dataframe):
        self.X = [torch.tensor([vocab[char] for char in seq], dtype=torch.long) for seq in dataframe['sequence']]
        self.Y = torch.tensor(dataframe['target_cost'].values, dtype=torch.float32)

    def __len__(self): return len(self.X)
    def __getitem__(self, idx): return self.X[idx], self.Y[idx]

def collate_fn(batch):
    X, Y = zip(*batch)
    lengths = torch.tensor([len(x) for x in X], dtype=torch.long)
    X_pad = pad_sequence(X, batch_first=True, padding_value=0)
    return X_pad, torch.stack(Y), lengths

dataset = StepSequenceDataset(df)
train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size
train_dataset, test_dataset = torch.utils.data.random_split(dataset, [train_size, test_size])

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, collate_fn=collate_fn)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, collate_fn=collate_fn)

# ==========================================
# 3. MODEL 
# ==========================================
class CostPredictorRNN(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super(CostPredictorRNN, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.rnn = nn.RNN(embed_dim, hidden_dim, batch_first=True, nonlinearity='relu')
        self.linear = nn.Linear(hidden_dim, 1)

    def forward(self, x, lengths):
        emb = self.embedding(x)
        packed_emb = nn.utils.rnn.pack_padded_sequence(emb, lengths, batch_first=True, enforce_sorted=False)
        _, h_n = self.rnn(packed_emb)
        prediction = self.linear(h_n[-1, :, :])
        return prediction.squeeze()

model = CostPredictorRNN(vocab_size=4, embed_dim=8, hidden_dim=32)
criterion = nn.L1Loss() 
optimizer = optim.Adam(model.parameters(), lr=0.01)
scheduler = lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5, verbose=True)

# ==========================================
# 4. TRAINING
# ==========================================
print("\nStarting training...")

for epoch in range(EPOCHS):
    model.train()
    total_train_loss = 0
    for X_batch, Y_batch, lengths in train_loader:
        optimizer.zero_grad()
        predictions = model(X_batch, lengths)
        if predictions.dim() == 0: predictions = predictions.unsqueeze(0)
        loss = criterion(predictions, Y_batch)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_train_loss += loss.item()
    
    avg_train_loss = total_train_loss / len(train_loader)
    
    model.eval()
    total_test_loss = 0
    with torch.no_grad():
        for X_batch_test, Y_batch_test, lengths_test in test_loader:
            test_predictions = model(X_batch_test, lengths_test)
            if test_predictions.dim() == 0: test_predictions = test_predictions.unsqueeze(0)
            test_loss = criterion(test_predictions, Y_batch_test)
            total_test_loss += test_loss.item()
            
    avg_test_loss = total_test_loss / len(test_loader)
    scheduler.step(avg_test_loss)
    
    if (epoch + 1) % 10 == 0 or epoch == 0:
        current_lr = optimizer.param_groups[0]['lr']
        print(f"Epoch {epoch+1:03d}/{EPOCHS} - Train MAE: {avg_train_loss:.4f} | Test MAE: {avg_test_loss:.4f} | LR: {current_lr:.6f}")

# ==========================================
# 5. TESTING
# ==========================================
print("\nTesting predictions (Network vs Reality):")
model.eval()
examples = ["A", "AB", "ABC", "CCC", "CCB", "AAB", "CBACBACBA"]

with torch.no_grad():
    for seq in examples:
        tensor_in = torch.tensor([vocab[c] for c in seq], dtype=torch.long).unsqueeze(0)
        length = torch.tensor([len(seq)], dtype=torch.long)
        
        pred = model(tensor_in, length).item()
        
        # Get the real clean cost for comparison
        _, clean_real = calculate_sequence_costs(seq, apply_noise=False)
        
        print(f"Sequence: {seq:<9} | Truth: {clean_real:5.1f} | RNN Prediction: {pred:5.1f}")

In [ ]:
#WE LEARN A DWFA FROM THE NETWORK

import torch
import sys
import os

from QuantitativeObservationTable import (
    QuantitativeObservationTable, 
    QuantitativeObservationTableParameters
)

from TrainedRNNOracle import TrainedRNNOracle

oracle = TrainedRNNOracle(rnn_model=model, vocab_dict=vocab, tol=0.5, rounding_step=1)

params = QuantitativeObservationTableParameters(
    tol_dist_init=0.7, 
    rec_method="direct", 
    r=3, 
)

table = QuantitativeObservationTable(
    alphabet=oracle.alphabet,
    membership_query=oracle.calculate_weight, 
    parameters=params
)

seen_counterexamples = set()
iteration = 1

while True:
    print(f"\n--- Iteration {iteration} ---")
    table.make_table_closed_and_consistent_det()
            
    # Reconstruct Hypothesis WFA
    wfa = table.reconstruct_deterministic_wfa_opt()
    wfa.check_twins_property()
    
    # Equivalence Query
    ce = oracle.equivalence_query(
        wfa, 
        method="hybrid", 
        exhaustive_len=6, 
        random_max_len=15, 
        num_random=1000,
    )

    # Evaluate Equivalence Query Results
    if ce is None:
        print(f"\n[SUCCESS] No counterexamples found.")
        print(f"Extraction complete in {iteration} iterations. Total states: {len(table.S)}")
        break

    if ce in seen_counterexamples:
        print(f"\n[STOP] Repeated counterexample '{ce}' detected. Check tolerances or network precision.")
        break

    print(f"[+] Processing Counterexample: '{ce}' | S size: {len(table.S)}")

    # Process Counterexample
    table.process_counterexample(ce, hypothesis_wfa=wfa, method="all_suffixes")
    seen_counterexamples.add(ce)
            
    iteration += 1

In [ ]:
#WE MANIPULATE THE WEIGHTS AS EXPLAINED IN THE THESIS

wfa.push_weights_to_positive()
wfa.normalize_global_bias()
wfa.push_final_weights_to_zero()

In [ ]:
#WE DRAW THE LEARNED DWFA AND DETECT THE SHORTCUT LEARNED BY THE RNN

dot = draw_tropical_wfa(wfa)
dot.render('Case1_deffect_wfa', view=True,format='png')

In [ ]:
#WE GENERATE A NOISY SYNTHETIC DATASET AND TRAIN AN RNN FROM IT

import random
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
import torch.optim.lr_scheduler as lr_scheduler

# ==========================================
# 0. CONFIGURATION
# ==========================================
APPLY_NOISE = True      
NOISE_BASE = 0.2         # Base minimum noise
NOISE_FACTOR = 0.15      # Noise scaling based on operation weight
DATASET_SIZE = 10000     
EPOCHS = 100            

# ==========================================
# 1. DATASET GENERATION
# ==========================================
def calculate_sequence_costs(sequence, apply_noise=True, noise_base=0.2, noise_factor=0.15):
    """
    Calculates the cost of a sequence. If apply_noise is True, it simulates 
    stochastic delays or advances based on the provided noise parameters.
    """
    total_clean_cost = 0.0     # The pure mathematical sum
    total_noisy_cost = 0.0     # The actual clocked time (with or without noise)
    previous = None
    
    for char in sequence:
        # 1. Determine the theoretical mean (mu) of this transition
        if char == 'A':
            mu = 1.0
        elif char == 'B':
            mu = 0.0 if previous == 'A' else 2.0
        elif char == 'C':
            mu = 10.0 if previous == 'C' else 5.0
            
        # 2. Apply noise if enabled
        if apply_noise:
            sigma = noise_base + (noise_factor * mu)
            operation_time = random.gauss(mu, sigma)
            operation_time = max(0.0, operation_time) # No negative time
        else:
            operation_time = mu
        
        # 3. Accumulate
        total_clean_cost += mu
        total_noisy_cost += operation_time
        previous = char
        
    return total_noisy_cost, total_clean_cost

data = []
for _ in range(DATASET_SIZE):
    length = random.randint(1, 10)
    sequence = "".join(random.choices(['A', 'B', 'C'], k=length))
    
    # Get costs using the defined configuration
    c_noisy, c_clean = calculate_sequence_costs(
        sequence, 
        apply_noise=APPLY_NOISE, 
        noise_base=NOISE_BASE, 
        noise_factor=NOISE_FACTOR
    )
    
    data.append({
        'sequence': sequence, 
        'target_cost': c_noisy,  # RNN Target (will be clean if noise is False)
        'clean_real_cost': c_clean 
    })

df = pd.DataFrame(data)

print(f"Dataset Sample (Noise Applied: {APPLY_NOISE}):")
# Use print(df.head(7)) if you are running this in a standard terminal
try:
    display(df.head(7))
except NameError:
    print(df.head(7))

# ==========================================
# 2. DATA PREPARATION
# ==========================================
vocab = {'A': 1, 'B': 2, 'C': 3}

class StepSequenceDataset(Dataset):
    def __init__(self, dataframe):
        self.X = [torch.tensor([vocab[char] for char in seq], dtype=torch.long) for seq in dataframe['sequence']]
        self.Y = torch.tensor(dataframe['target_cost'].values, dtype=torch.float32)

    def __len__(self): return len(self.X)
    def __getitem__(self, idx): return self.X[idx], self.Y[idx]

def collate_fn(batch):
    X, Y = zip(*batch)
    lengths = torch.tensor([len(x) for x in X], dtype=torch.long)
    X_pad = pad_sequence(X, batch_first=True, padding_value=0)
    return X_pad, torch.stack(Y), lengths

dataset = StepSequenceDataset(df)
train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size
train_dataset, test_dataset = torch.utils.data.random_split(dataset, [train_size, test_size])

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, collate_fn=collate_fn)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, collate_fn=collate_fn)

# ==========================================
# 3. MODEL 
# ==========================================
class CostPredictorRNN(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super(CostPredictorRNN, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.rnn = nn.RNN(embed_dim, hidden_dim, batch_first=True, nonlinearity='relu')
        self.linear = nn.Linear(hidden_dim, 1)

    def forward(self, x, lengths):
        emb = self.embedding(x)
        packed_emb = nn.utils.rnn.pack_padded_sequence(emb, lengths, batch_first=True, enforce_sorted=False)
        _, h_n = self.rnn(packed_emb)
        prediction = self.linear(h_n[-1, :, :])
        return prediction.squeeze()

model = CostPredictorRNN(vocab_size=4, embed_dim=8, hidden_dim=32)
criterion = nn.L1Loss() 
optimizer = optim.Adam(model.parameters(), lr=0.01)
scheduler = lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5, verbose=True)

# ==========================================
# 4. TRAINING
# ==========================================
print("\nStarting training...")

for epoch in range(EPOCHS):
    model.train()
    total_train_loss = 0
    for X_batch, Y_batch, lengths in train_loader:
        optimizer.zero_grad()
        predictions = model(X_batch, lengths)
        if predictions.dim() == 0: predictions = predictions.unsqueeze(0)
        loss = criterion(predictions, Y_batch)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_train_loss += loss.item()
    
    avg_train_loss = total_train_loss / len(train_loader)
    
    model.eval()
    total_test_loss = 0
    with torch.no_grad():
        for X_batch_test, Y_batch_test, lengths_test in test_loader:
            test_predictions = model(X_batch_test, lengths_test)
            if test_predictions.dim() == 0: test_predictions = test_predictions.unsqueeze(0)
            test_loss = criterion(test_predictions, Y_batch_test)
            total_test_loss += test_loss.item()
            
    avg_test_loss = total_test_loss / len(test_loader)
    scheduler.step(avg_test_loss)
    
    if (epoch + 1) % 10 == 0 or epoch == 0:
        current_lr = optimizer.param_groups[0]['lr']
        print(f"Epoch {epoch+1:03d}/{EPOCHS} - Train MAE: {avg_train_loss:.4f} | Test MAE: {avg_test_loss:.4f} | LR: {current_lr:.6f}")

# ==========================================
# 5. TESTING
# ==========================================
print("\nTesting predictions (Network vs Reality):")
model.eval()
examples = ["A", "AB", "ABC", "CCC", "CCB", "AAB", "CBACBACBA"]

with torch.no_grad():
    for seq in examples:
        tensor_in = torch.tensor([vocab[c] for c in seq], dtype=torch.long).unsqueeze(0)
        length = torch.tensor([len(seq)], dtype=torch.long)
        
        pred = model(tensor_in, length).item()
        
        # Get the real clean cost for comparison
        _, clean_real = calculate_sequence_costs(seq, apply_noise=False)
        
        print(f"Sequence: {seq:<9} | Truth: {clean_real:5.1f} | RNN Prediction: {pred:5.1f}")

In [ ]:
#WE LEARN A DWFA FROM THE NETWORK

import torch
import sys
import os

from QuantitativeObservationTable import (
    QuantitativeObservationTable, 
    QuantitativeObservationTableParameters
)

from TrainedRNNOracle import TrainedRNNOracle

oracle = TrainedRNNOracle(rnn_model=model, vocab_dict=vocab, tol=0.5, rounding_step=0)

params = QuantitativeObservationTableParameters(
    tol_dist_init=1, 
    rec_method="direct", 
    r=3, 
)

table = QuantitativeObservationTable(
    alphabet=oracle.alphabet,
    membership_query=oracle.calculate_weight, 
    parameters=params
)

seen_counterexamples = set()
iteration = 1

while True:
    print(f"\n--- Iteration {iteration} ---")
    table.make_table_closed_and_consistent_det()
            
    # Reconstruct Hypothesis WFA
    wfa = table.reconstruct_deterministic_wfa_opt()
    wfa.check_twins_property()
    
    # Equivalence Query
    ce = oracle.equivalence_query(
        wfa, 
        method="hybrid", 
        exhaustive_len=4, 
        random_max_len=15, 
        num_random=0,
    )

    # Evaluate Equivalence Query Results
    if ce is None:
        print(f"\n[SUCCESS] No counterexamples found.")
        print(f"Extraction complete in {iteration} iterations. Total states: {len(table.S)}")
        break

    if ce in seen_counterexamples:
        print(f"\n[STOP] Repeated counterexample '{ce}' detected. Check tolerances or network precision.")
        break

    print(f"[+] Processing Counterexample: '{ce}' | S size: {len(table.S)}")

    # Process Counterexample
    table.process_counterexample(ce, hypothesis_wfa=wfa, method="all_suffixes")
    seen_counterexamples.add(ce)
            
    iteration += 1

In [ ]:
#WE MANIPULATE THE WEIGHTS AS EXPLAINED IN THE THESIS

wfa.push_weights_to_positive()
wfa.normalize_global_bias()
wfa.push_final_weights_to_zero()

In [ ]:
#WE DRAW THE LEARNED WFA

dot = draw_tropical_wfa(wfa)
dot.render('Case1_noisy_wfa3', view=True,format='png')